[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_04_data_science_math/ch16_numpy_vectorization/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 16 Walkthrough: NumPy & Vectorized Thinking

**Book Reference:** Chapter 16, Sections 16.1 – 16.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 16.3.1 — Contiguous Arrays vs. Python Lists

In [ ]:
import numpy as np

w_list = [64.5, 54.6, 70.0, 81.2]
h_list = [168.0, 156.0, 175.0, 180.0]

weights = np.array(w_list, dtype=float)
heights = np.array(h_list, dtype=float)

# List `* 2` repeats the container; NumPy `* 2` scales every value!
print("List * 2: ", w_list[:2] * 2)
print("Array * 2:", weights[:2] * 2)

# Vectorized BMI in one C-speed expression (zero Python for-loops)
bmi = np.round(weights / ((heights / 100.0) ** 2), 2)
print("Cohort BMI:", bmi)
print(
    f"type={type(bmi).__name__}, shape={bmi.shape}, "
    f"dtype={bmi.dtype}, nbytes={bmi.nbytes}"
)


## Section 16.3.2 — Shapes, Axes (`axis=0` vs. `axis=1`) & Dtypes

In [ ]:
# 3 districts (rows, axis=0) x 4 weeks (columns, axis=1)
cases = np.array([
    [12, 25, 40, 23],  # Dhaka
    [30, 15, 10,  5],  # Sylhet
    [ 5,  8, 12, 20],  # Khulna
])

print("shape:", cases.shape, "| ndim:", cases.ndim, "| dtype:", cases.dtype)
print("Total per district (axis=1):", np.sum(cases, axis=1))
print("Weekly mean (axis=0):       ", np.round(np.mean(cases, axis=0), 2))
print("Peak week index (axis=1):   ", np.argmax(cases, axis=1))

# Homogeneous dtype upcasting: mixing int + float promotes to float64
mixed = np.array([10, 11.5, 12])
print("Upcast mixed dtype:         ", mixed.dtype, mixed)


## Section 16.3.3 — The View-vs-Copy Trap (Slicing Shares Memory!)

In [ ]:
readings = np.array([10.4, 11.2, 12.1, 13.0, 11.8])

# NumPy slice = VIEW (shares underlying C memory buffer!)
view_slice = readings[1:4]
safe_copy = readings[1:4].copy()

print("Shares memory (view)?", np.shares_memory(readings, view_slice))
print("Shares memory (copy)?", np.shares_memory(readings, safe_copy))

view_slice[0] = -1.0  # Silently mutates `readings[1]`!
print("Original after view edit:", readings)

safe_copy[0] = 99.0   # Does NOT touch `readings`
print("Original after copy edit:", readings)


## Section 16.3.4 — Boolean Masking & Sentinel-Safe `np.nanmean`

In [ ]:
raw_hb = np.array([10.4, 999.0, 11.8, np.nan, 10.6, 12.6])

# Always copy as float before inserting np.nan!
clean_hb = np.array(raw_hb, dtype=float, copy=True)
clean_hb[clean_hb == 999.0] = np.nan

print("np.mean(clean_hb):   ", np.mean(clean_hb))
print("np.nanmean(clean_hb):", round(float(np.nanmean(clean_hb)), 2))

anemic_mask = clean_hb < 11.0
print("Anemic mask:         ", anemic_mask)
print("Anemic count:        ", int(np.sum(anemic_mask)))
print("Raw input untouched: ", raw_hb)


## Section 16.3.5 — Broadcasting Rules in Action

In [ ]:
# Feature matrix: 3 patients x 2 biomarkers [hb_g_dl, sbp_mmhg]
X = np.array([
    [10.0, 120.0],
    [12.0, 130.0],
    [14.0, 140.0],
])

# 1. Column centering: (3, 2) - (2,) broadcasts across all 3 rows
col_means = np.mean(X, axis=0)
centered = X - col_means
print("Column means (2,):", col_means)
print("Centered (3, 2):\n", centered)

# 2. Row-wise scaling via [:, None]: (3, 4) / (3, 1)
pop = np.array([250_000, 120_000, 90_000], dtype=float)
weekly_inc = np.round((cases / pop[:, None]) * 100_000, 2)
print("pop[:, None] shape:", pop[:, None].shape)
print("Weekly incidence per 100k:\n", weekly_inc)


## Section 16.3.6 — Linear Algebra in Action (`@`, Sigmoid & Norms)

In [ ]:
Z = np.array([
    [-1.2,  0.8],
    [ 0.0,  0.0],
    [ 1.5, -0.5],
])
w = np.array([-0.9, 1.1])
b = 0.2

# Linear logits (3, 2) @ (2,) -> (3,), then logistic sigmoid
logits = Z @ w + b
probs = 1.0 / (1.0 + np.exp(-logits))
print("Logits:", np.round(logits, 4))
print("Probs: ", np.round(probs, 4))

# Euclidean (L2) distances from healthy centroid [0.0, 0.0]
ref = np.array([0.0, 0.0])
dists = np.round(np.linalg.norm(Z - ref, axis=1), 4)
print("L2 distances to ref:", dists)


## Section 16.3.7 — Break It On Purpose! (NumPy Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — incompatible broadcasting shapes
mat = np.ones((3, 2))
vec = np.array([1, 2, 3, 4])
bad_sum = mat + vec


In [ ]:
# Intentional Bug #2: ValueError — misaligned matrix multiply (@)
A = np.ones((3, 2))
B = np.ones((3, 1))
bad_prod = A @ B


In [ ]:
# Intentional Bug #3: IndexError — out-of-bounds 2D axis index
grid = np.zeros((3, 4))
bad_cell = grid[3, 0]


## Section 16.4 — Worked Problem-Structuring Blueprint: Vectorized Clinical Risk Pipeline

In [ ]:
def run_vectorized_triage(raw_X, weights, bias, cutoff=0.5):
    """Clean, impute, standardize, and score a vital matrix."""
    # Step 1: Copy as float & replace sentinel 999.0 with np.nan
    X = np.array(raw_X, dtype=float, copy=True)
    X[X == 999.0] = np.nan

    # Step 2: Broadcasted column-mean imputation
    col_means = np.nanmean(X, axis=0)
    X = np.where(np.isnan(X), col_means, X)

    # Step 3: Column Z-score standardization with zero-std guard
    means = np.mean(X, axis=0)
    stds = np.std(X, axis=0)
    safe_stds = np.where(stds == 0, 1.0, stds)
    Z = (X - means) / safe_stds
    Z[:, stds == 0] = 0.0

    # Step 4: Linear logits (@), sigmoid probs, and 0/1 flags
    logits = Z @ np.asarray(weights, dtype=float) + bias
    probs = 1.0 / (1.0 + np.exp(-logits))
    flags = (probs >= cutoff).astype(int)
    return np.round(probs, 4), flags

# 4 patients x 3 features: [hb_g_dl, temp_c, clinic_code]
raw_vitals = np.array([
    [ 9.5, 39.1, 1.0],
    [999.0, 38.4, 1.0],
    [12.5, 36.8, 1.0],
    [11.0, np.nan, 1.0],
])
w_triage = np.array([-1.2, 1.5, 0.5])
probs, flags = run_vectorized_triage(raw_vitals, w_triage, bias=0.1)

print("Risk probabilities:", probs)
print("Triage flags (0/1):", flags)
print("Raw sentinel intact:", raw_vitals[1, 0])
